# 대화 이력을 기억하는 RAG — 문맥을 유지하는 대화형 QA

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- `RunnableWithMessageHistory`로 대화 이력을 RAG 체인에 통합하기
- `ChatMessageHistory`와 세션 ID로 여러 사용자의 대화를 독립 관리하기
- `MessagesPlaceholder`를 프롬프트에 삽입해 이전 대화를 문맥으로 활용하기

## 사전 지식

- 00-RAG-Basic.ipynb의 8단계 RAG 파이프라인
- 세션(Session): 사용자별 독립적인 대화 공간

---

```mermaid
flowchart LR
    U[사용자 질문]:::input --> H[ChatMessageHistory<br/>이전 대화 로드]:::storage
    H --> P[ChatPromptTemplate<br/>system + history + question]:::process
    R[(VectorStore)]:::storage --> P
    P --> L[LLM<br/>문맥 있는 답변]:::process
    L --> S[ChatMessageHistory<br/>답변 저장]:::storage
    L --> A[최종 답변]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

## 일반 RAG vs 대화형 RAG

| | 일반 RAG | 대화형 RAG |
|--|---------|-----------|
| 질문 1 | "인공지능이 뭐야?" → 잘 답변 | "인공지능이 뭐야?" → 잘 답변 |
| 질문 2 | "그것의 역사는?" → "그것"이 뭔지 모름 | "그것의 역사는?" → 인공지능의 역사로 이해 |

> **실무 팁**: 대화가 길어지면 컨텍스트 토큰이 증가해요. 최근 5~10개 메시지만 유지하거나, 오래된 대화를 요약해서 저장하는 전략을 고려해보세요.

> **주의**: `RUN_CONVERSATION_DEMO=1` 환경변수를 설정해야 대화형 RAG 예제가 실행돼요. API 비용 제어를 위한 설정입니다.

## 일반 RAG vs 대화형 RAG

### 일반 RAG의 한계

```
User: 인공지능이 뭐야?
AI: 인공지능은 컴퓨터가 인간처럼 생각하고 학습하는 기술입니다.

User: 그것의 역사는?
AI: ❌ "그것"이 무엇을 가리키는지 모름
```

### 대화형 RAG

```
User: 인공지능이 뭐야?
AI: 인공지능은 컴퓨터가 인간처럼 생각하고 학습하는 기술입니다.

User: 그것의 역사는?
AI: ✅ 인공지능의 역사는 1950년대에 시작되었습니다...
     (이전 대화에서 "그것" = "인공지능" 파악)
```

In [ ]:
import os
from dotenv import load_dotenv

# 아래에 코드를 작성하세요


In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory

## 1. 문서 준비 및 RAG 기본 설정

In [ ]:
# ---------------------------------------------------
# 문서 로드, 분할, 임베딩, 벡터스토어 초기화
# ---------------------------------------------------
# ============================================================
# TODO: PDF를 로드하고 분할한 뒤 FAISS 벡터스토어와 검색기를 만드세요
# 힌트: PyMuPDFLoader → split_documents → FAISS.from_documents → as_retriever(k=4)
# 예상 결과: "문서 및 검색기 준비 완료" 출력
# ============================================================

# 문서 로드

# 분할

# 벡터스토어

# 검색기


In [ ]:
# ---------------------------------------------------
# 세션별 대화 이력 저장소 구성
# ---------------------------------------------------
# ============================================================
# TODO: get_session_history() 함수를 완성하여 session_id별로 이력을 반환하세요
# 힌트: store 딕셔너리에 ChatMessageHistory()를 세션별로 저장
# 예상 결과: "대화 이력 저장소 설정 완료" 출력
# ============================================================

# 세션별 대화 이력 저장소 (인메모리)
# ⚠️ 주의: 서버 재시작 시 모든 대화 이력이 사라짐 — 프로덕션에선 Redis/DB 사용

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """세션 ID에 해당하는 대화 이력을 반환합니다."""
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


In [ ]:
# ---------------------------------------------------
# 세션별 대화 이력 저장소 구성
# ---------------------------------------------------
# ============================================================
# TODO: get_session_history() 함수를 완성하여 session_id별로 이력을 반환하세요
# 힌트: store 딕셔너리에 ChatMessageHistory()를 세션별로 저장
# 예상 결과: "대화 이력 저장소 설정 완료" 출력
# ============================================================

# 세션별 대화 이력 저장소 (인메모리)

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """세션 ID에 해당하는 대화 이력을 반환합니다."""
    # 처음 요청된 세션이면 새 ChatMessageHistory 생성
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


In [ ]:
# ---------------------------------------------------
# MessagesPlaceholder로 대화 이력이 들어가는 프롬프트 구성
# ---------------------------------------------------
# ============================================================
# TODO: ChatPromptTemplate에 MessagesPlaceholder("chat_history")를 삽입하세요
# 힌트: from_messages([("system", ...), MessagesPlaceholder("chat_history"), ("human", ...)])
# 예상 결과: "대화형 프롬프트 생성 완료" 출력
# ============================================================

# system → [이전 대화들] → human 순서가 LLM이 문맥을 올바르게 파악하는 핵심 구조


#Context:


In [ ]:
# ---------------------------------------------------
# MessagesPlaceholder로 대화 이력이 들어가는 프롬프트 구성
# ---------------------------------------------------
# ============================================================
# TODO: ChatPromptTemplate에 MessagesPlaceholder("chat_history")를 삽입하세요
# 힌트: from_messages([("system", ...), MessagesPlaceholder("chat_history"), ("human", ...)])
# 예상 결과: "대화형 프롬프트 생성 완료" 출력
# ============================================================

# MessagesPlaceholder: 런타임에 실제 대화 메시지 목록으로 교체되는 자리표시자


#Context:


In [ ]:
# ---------------------------------------------------
# RunnableWithMessageHistory로 대화 이력 관리를 체인에 통합
# ---------------------------------------------------
# ============================================================
# TODO: rag_chain을 RunnableWithMessageHistory로 감싸세요
# 힌트: RunnableWithMessageHistory(rag_chain, get_session_history, input_messages_key="question", history_messages_key="chat_history")
# 예상 결과: "대화형 RAG 체인 생성 완료" 출력
# ============================================================



        # 이력을 로드하고 저장하는 래퍼(Wrapper) 역할을 합니다
        # input_messages_key: 사용자 입력 키 / history_messages_key: 프롬프트 변수와 일치해야 함


In [ ]:
# ---------------------------------------------------
# RunnableWithMessageHistory로 대화 이력 관리를 체인에 통합
# ---------------------------------------------------
# ============================================================
# TODO: rag_chain을 RunnableWithMessageHistory로 감싸세요
# 힌트: RunnableWithMessageHistory(rag_chain, get_session_history, input_messages_key="question", history_messages_key="chat_history")
# 예상 결과: "대화형 RAG 체인 생성 완료" 출력
# ============================================================

        # LLM

        # 기본 RAG 체인 (대화 이력 키 포함)

        # RunnableWithMessageHistory: 체인을 감싸 대화 이력을 자동으로 읽고 저장


## 5. 대화형 RAG 실행

세션 ID를 지정하여 대화를 시작합니다.

In [ ]:
    # 세션 설정

    # 첫 번째 질문

# 아래에 코드를 작성하세요


In [ ]:
    # 두 번째 질문 (이전 대화 참조)

# 아래에 코드를 작성하세요


In [ ]:
    # 세 번째 질문 (추가 정보)

# 아래에 코드를 작성하세요


## 6. 대화 이력 확인

In [ ]:
    # 현재 세션의 대화 이력 출력

# 아래에 코드를 작성하세요


## 7. 다중 세션 테스트

서로 다른 세션은 독립적인 대화 이력을 유지합니다.

In [ ]:
    # 두 번째 사용자 (다른 세션)



    # user_001의 이력은 유지됨

# 아래에 코드를 작성하세요


## 8. 대화 이력 초기화

In [ ]:
# 특정 세션 초기화
def clear_session(session_id: str):
    """세션의 대화 이력을 초기화합니다."""
    if session_id in store:
        store[session_id].clear()
        print(f"✅ 세션 '{session_id}' 초기화 완료")
    else:
        print(f"⚠️  세션 '{session_id}'이(가) 존재하지 않습니다.")

# 모든 세션 초기화
def clear_all_sessions():
    """모든 세션의 대화 이력을 초기화합니다."""
    store.clear()
    print("✅ 모든 세션 초기화 완료")

# 테스트

# 아래에 코드를 작성하세요


## 💡 핵심 정리

### 대화형 RAG의 구성 요소

1. **ChatMessageHistory**: 대화 이력 저장
2. **MessagesPlaceholder**: 프롬프트에 이력 삽입
3. **RunnableWithMessageHistory**: 체인에 이력 관리 추가
4. **세션 관리**: 사용자별 독립적인 대화 유지

### 실전 활용 시나리오

- **고객 지원 챗봇**: 고객과의 대화 문맥 유지
- **교육 AI 튜터**: 학습 진행 상황 기억
- **문서 탐색 도우미**: 연속된 질문으로 심화 탐구

### 대화 이력 저장 옵션

**메모리 기반 (현재)**:
```python
store = {}  # 재시작 시 초기화
```

**영구 저장 (Production)**:
```python
# SQLite, Redis, PostgreSQL 등 사용
from langchain.memory import SQLChatMessageHistory

def get_session_history(session_id: str):
    return SQLChatMessageHistory(
        session_id=session_id,
        connection_string="sqlite:///chat_history.db"
    )
```

### 주의사항

- ⚠️ **컨텍스트 길이**: 대화가 길어지면 토큰 제한 초과 가능
  - 해결: 최근 N개 메시지만 유지하거나 요약 사용
- ⚠️ **개인정보**: 대화 이력에 민감 정보 포함 주의
  - 해결: 암호화 저장, 주기적 삭제

### 성능 향상 팁

1. **대화 길이 제한**: 최근 5~10개 메시지만 유지
2. **요약 활용**: 긴 대화는 요약하여 저장
3. **세션 타임아웃**: 일정 시간 후 자동 초기화

---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 구성 요소 | `RunnableWithMessageHistory` + `ChatMessageHistory` |
| 세션 관리 | `session_id`로 여러 사용자/대화를 독립적으로 관리 |
| 이력 저장 | 인메모리 딕셔너리 — 프로덕션에선 Redis/DB로 교체 권장 |
| `MessagesPlaceholder` | 프롬프트 내 대화 이력 삽입 위치 지정 |
| 주의 | 이력이 길어질수록 토큰 비용 증가 — 최근 N턴만 유지하는 전략 고려 |

### 대화형 RAG 핵심 흐름

```
1. 사용자 질문 입력
2. ChatMessageHistory에서 해당 session_id의 이력 로드
3. 이력 + 현재 질문으로 VectorStore 검색
4. 검색 결과 + 이력 + 질문 → LLM 답변 생성
5. 새 질문/답변 쌍을 ChatMessageHistory에 저장
```

### 세션 이력 관리 옵션

| 저장소 | 용도 | 장점 |
|--------|------|------|
| `ChatMessageHistory` (기본) | 단일 서버, 개발용 | 간단, 설치 불필요 |
| `RedisChatMessageHistory` | 멀티 서버, 프로덕션 | 영속성, 분산 지원 |
| `SQLChatMessageHistory` | DB 기반 저장 | 이력 조회·분석 가능 |

### 다음 노트북 예고

**04-RAPTOR** — 긴 문서를 계층적으로 요약해 레벨별 검색이 가능한 트리 구조를 만드는 RAPTOR(Recursive Abstractive Processing for Tree-Organized Retrieval) 기법을 배워요.